# 01 · Data Cleaning — Session Export

**Client:** a mid-sized Brazilian online retailer (fashion & home goods, São Paulo). Name withheld for confidentiality.
**Business question:** *Which website sessions are likely to end in a purchase, so marketing can act while the visitor is still on site?*

This notebook takes the **raw session export** handed over by the client's analytics team and turns it into a clean,
analysis-ready table. The raw file mixes Portuguese/English labels, two date formats, BRL text values, duplicated
rows from a double export and bot traffic.

> ⚠️ Client name withheld; the dataset is **synthetic**, generated by `src/generate_raw_data.py`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
RAW = "../data/raw/sessions_raw.csv"
OUT = "../data/processed/sessions_clean.csv"

raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(16202, 22)


,session_id,session_start,customer_state,device_type,traffic_source,visitor_type,pages_viewed,category_pages,product_pages,searches,session_duration_sec,product_page_duration_sec,add_to_cart_count,wishlist_adds,cart_value_brl,viewed_freight_calculator,viewed_installment_options,coupon_applied,bounce_rate,exit_rate,page_value_brl,purchased
0,CE-1673761,2026-09-02 17:56:36,SP,Tablet,Google Ads,Novo,18,6,9,1,159.0,103.0,1,0,300.56,não,não,sim,0.0304,0.2412,92.08,1
1,CE-3275967,2026-09-18 12:06:16,mg,mobile,Instagram,Recorrente,7,2,4,0,200.0,114.0,0,0,0.0,não,0,não,0.3466,0.3942,0.00,0
2,CE-1475394,2026-08-12 05:58:34,sp,Mobile,Google Orgânico,Novo,7,2,4,0,380.0,215.0,0,0,"R$ 0,00",não,sim,não,0.1931,0.4585,0.00,não
3,CE-1640719,30/09/2026 20:28,DF,Mobile,Google Ads,Novo,8,3,0,2,566.0,94.0,0,0,0.0,não,False,não,0.2406,0.3178,0.00,não
4,CE-6957740,26/04/2026 17:23,RJ,Mobile,Google Orgânico,Recorrente,2,0,1,1,50.0,7.0,0,0,0.0,False,não,0,0.9052,0.7677,0.00,não


## 1. First look

In [2]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 16202 entries, 0 to 16201
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   session_id                  16202 non-null  str    
 1   session_start               16202 non-null  str    
 2   customer_state              15861 non-null  str    
 3   device_type                 16073 non-null  str    
 4   traffic_source              15975 non-null  str    
 5   visitor_type                16107 non-null  str    
 6   pages_viewed                16202 non-null  int64  
 7   category_pages              16202 non-null  int64  
 8   product_pages               16202 non-null  int64  
 9   searches                    16202 non-null  int64  
 10  session_duration_sec        15705 non-null  float64
 11  product_page_duration_sec   15764 non-null  float64
 12  add_to_cart_count           16202 non-null  int64  
 13  wishlist_adds               16202 non-null

In [3]:
raw.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)

,0
session_duration_sec,497
product_page_duration_sec,438
customer_state,341
traffic_source,227
exit_rate,196
device_type,129
visitor_type,95


In [4]:
for col in ["customer_state", "device_type", "purchased", "viewed_freight_calculator"]:
    print(col, "→", raw[col].dropna().unique()[:15], "\n")

customer_state → <StringArray>
[  'SP',   'mg',   'sp',   'DF',   'RJ',   'MG',   'ES',   'AM',   'BA',
   'GO',   'CE',   'SC',   'PR',   'rs', ' DF ']
Length: 15, dtype: str 

device_type → <StringArray>
[    'Tablet',     'mobile',     'Mobile',    'Desktop',     'tablet',
    'Celular',     'MOBILE',    'desktop', 'Computador']
Length: 9, dtype: str 

purchased → <StringArray>
['1', '0', 'não', 'sim']
Length: 4, dtype: str 

viewed_freight_calculator → <StringArray>
['não', 'False', 'sim', '0', 'True', '1']
Length: 6, dtype: str 



**Issues found**

| # | Issue | Fix |
|---|---|---|
| 1 | Exact duplicate rows (double export) | drop duplicates |
| 2 | Dates in two formats (`2026-03-01 14:05:10` and `01/03/2026 14:05`) | parse each format explicitly |
| 3 | State as `sp`, ` SP `, `São Paulo` | map to 2-letter UF code |
| 4 | Device as `Mobile / Celular / MOBILE …` | map to Mobile / Desktop / Tablet |
| 5 | Booleans as `sim/não`, `1/0`, `True/False` | map to 0/1 |
| 6 | `cart_value_brl` partly text like `R$ 1.234,56` | parse BR number format |
| 7 | Negative session durations (tracking glitch) | set to NaN, then impute |
| 8 | Bot sessions (hundreds of pages in a few seconds) | remove |
| 9 | Missing values in several columns | impute (see below) |

## 2. Duplicates

In [5]:
df = raw.copy()
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_dup} exact duplicates → {len(df):,} rows")
print("Duplicate session_id left:", df["session_id"].duplicated().sum())

Removed 211 exact duplicates → 15,991 rows
Duplicate session_id left: 0


## 3. Dates — two formats

In [6]:
iso = pd.to_datetime(df["session_start"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
br = pd.to_datetime(df["session_start"], format="%d/%m/%Y %H:%M", errors="coerce")
df["session_start"] = iso.fillna(br)
print("Unparsed dates:", df["session_start"].isna().sum())
print("Range:", df["session_start"].min(), "→", df["session_start"].max())

Unparsed dates: 0
Range: 2025-10-01 02:26:00 → 2026-09-30 23:38:00


## 4. Categorical labels

In [7]:
name_to_uf = {"são paulo": "SP", "rio de janeiro": "RJ", "minas gerais": "MG",
              "rio grande do sul": "RS", "bahia": "BA", "paraná": "PR"}
s = df["customer_state"].str.strip().str.lower()
df["customer_state"] = s.map(name_to_uf).fillna(s.str.upper())

device_map = {"mobile": "Mobile", "celular": "Mobile", "desktop": "Desktop",
              "computador": "Desktop", "tablet": "Tablet"}
df["device_type"] = df["device_type"].str.strip().str.lower().map(device_map)

df["visitor_type"] = df["visitor_type"].map({"Novo": "New", "Recorrente": "Returning"})

print(df["customer_state"].value_counts(dropna=False).head(8), "\n")
print(df["device_type"].value_counts(dropna=False))

customer_state
SP    4914
RJ    1853
MG    1740
PR    1070
RS    1052
BA     811
SC     777
GO     533
Name: count, dtype: int64 

device_type
Mobile     10727
Desktop     4339
Tablet       798
NaN          127
Name: count, dtype: int64


## 5. Booleans and BRL values

In [8]:
bool_map = {"sim": 1, "1": 1, "True": 1, "não": 0, "0": 0, "False": 0}
bool_cols = ["viewed_freight_calculator", "viewed_installment_options", "coupon_applied", "purchased"]
for c in bool_cols:
    df[c] = df[c].astype(str).map(bool_map).astype("int8")

def parse_brl(x):
    if isinstance(x, str) and x.startswith("R$"):
        return float(x.replace("R$", "").strip().replace(".", "").replace(",", "."))
    return float(x)

df["cart_value_brl"] = df["cart_value_brl"].map(parse_brl)
df[bool_cols + ["cart_value_brl"]].describe().T

,count,mean,std,min,25%,50%,75%,max
viewed_freight_calculator,15991.0,0.399099,0.489729,0.0,0.0,0.0,1.000,1.00
viewed_installment_options,15991.0,0.294978,0.456047,0.0,0.0,0.0,1.000,1.00
coupon_applied,15991.0,0.093115,0.290602,0.0,0.0,0.0,0.000,1.00
purchased,15991.0,0.171722,0.377150,0.0,0.0,0.0,0.000,1.00
cart_value_brl,15991.0,130.676267,264.904551,0.0,0.0,0.0,163.665,4021.31


## 6. Bots and impossible values
A human cannot view 180+ pages in under a minute. These sessions are crawlers and would distort both EDA and the model.

In [9]:
bot_mask = (df["pages_viewed"] > 150) | df["session_id"].str.contains("BOT")
print("Bot sessions removed:", bot_mask.sum())
df = df[~bot_mask].reset_index(drop=True)

neg = df["session_duration_sec"] < 0
print("Negative durations set to NaN:", neg.sum())
df.loc[neg, "session_duration_sec"] = np.nan

Bot sessions removed: 118
Negative durations set to NaN: 36


## 7. Missing values
* **Categoricals** (state, source, device, visitor type) → explicit `"Unknown"` category; *the fact that tracking is missing can itself carry information*.
* **Durations** → median **per number of pages viewed bucket** (longer sessions have more pages), not a single global median.
* **exit_rate** → median.

In [10]:
for c in ["customer_state", "traffic_source", "device_type", "visitor_type"]:
    df[c] = df[c].fillna("Unknown")

page_bucket = pd.cut(df["pages_viewed"], [0, 3, 6, 10, 20, 40, 200])
for c in ["session_duration_sec", "product_page_duration_sec"]:
    df[c] = df[c].fillna(df.groupby(page_bucket, observed=True)[c].transform("median"))

# product time can never exceed total time
df["product_page_duration_sec"] = np.minimum(df["product_page_duration_sec"], df["session_duration_sec"])
df["exit_rate"] = df["exit_rate"].fillna(df["exit_rate"].median())
print("Remaining NaN:", df.isna().sum().sum())

Remaining NaN: 0


## 8. Calendar features (Brazilian retail calendar)

In [11]:
events = [("2025-11-24", "2025-12-01", "Black Friday"), ("2025-12-10", "2025-12-23", "Natal"),
          ("2026-02-14", "2026-02-17", "Carnaval"), ("2026-03-09", "2026-03-15", "Semana do Consumidor"),
          ("2026-05-01", "2026-05-10", "Dia das Mães"), ("2026-06-05", "2026-06-12", "Dia dos Namorados")]
df["sales_event"] = "Regular day"
for start, end, name in events:
    m = df["session_start"].between(start, pd.Timestamp(end) + pd.Timedelta(days=1))
    df.loc[m, "sales_event"] = name

df["month"] = df["session_start"].dt.month
df["hour"] = df["session_start"].dt.hour
df["is_weekend"] = (df["session_start"].dt.dayofweek >= 5).astype("int8")
df["region"] = df["customer_state"].map({
    "SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste", "ES": "Sudeste",
    "PR": "Sul", "SC": "Sul", "RS": "Sul",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "BA": "Nordeste", "PE": "Nordeste", "CE": "Nordeste", "RN": "Nordeste",
    "PA": "Norte", "AM": "Norte"}).fillna("Unknown")
df["sales_event"].value_counts()

,count
sales_event,
Regular day,12338
Natal,998
Dia das Mães,725
Dia dos Namorados,599
Black Friday,564
Semana do Consumidor,521
Carnaval,128


## 9. Final checks & save

In [12]:
assert df["session_id"].is_unique
assert (df["session_duration_sec"] >= 0).all()
assert df.isna().sum().sum() == 0
print(f"Raw rows: {len(raw):,} → clean rows: {len(df):,}")
print(f"Purchase rate: {df['purchased'].mean():.1%}")

import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv(OUT, index=False)
df.head()

Raw rows: 16,202 → clean rows: 15,873
Purchase rate: 17.3%


,session_id,session_start,customer_state,device_type,traffic_source,visitor_type,pages_viewed,category_pages,product_pages,searches,session_duration_sec,product_page_duration_sec,add_to_cart_count,wishlist_adds,cart_value_brl,viewed_freight_calculator,viewed_installment_options,coupon_applied,bounce_rate,exit_rate,page_value_brl,purchased,sales_event,month,hour,is_weekend,region
0,CE-1673761,2026-09-02 17:56:36,SP,Tablet,Google Ads,New,18,6,9,1,159.0,103.0,1,0,300.56,0,0,1,0.0304,0.2412,92.08,1,Regular day,9,17,0,Sudeste
1,CE-3275967,2026-09-18 12:06:16,MG,Mobile,Instagram,Returning,7,2,4,0,200.0,114.0,0,0,0.00,0,0,0,0.3466,0.3942,0.00,0,Regular day,9,12,0,Sudeste
2,CE-1475394,2026-08-12 05:58:34,SP,Mobile,Google Orgânico,New,7,2,4,0,380.0,215.0,0,0,0.00,0,1,0,0.1931,0.4585,0.00,0,Regular day,8,5,0,Sudeste
3,CE-1640719,2026-09-30 20:28:00,DF,Mobile,Google Ads,New,8,3,0,2,566.0,94.0,0,0,0.00,0,0,0,0.2406,0.3178,0.00,0,Regular day,9,20,0,Centro-Oeste
4,CE-6957740,2026-04-26 17:23:00,RJ,Mobile,Google Orgânico,Returning,2,0,1,1,50.0,7.0,0,0,0.00,0,0,0,0.9052,0.7677,0.00,0,Regular day,4,17,1,Sudeste
